# Notebook 11: The Attention Mechanism
**Estimated time:** 40-50 min
**Prerequisites:** Notebooks 1-10

---
## What you will learn
1. The fundamental problem attention solves
2. Scaled dot-product attention — derived from scratch
3. What Query, Key, and Value actually mean
4. Self-attention vs cross-attention
5. Implementing attention from scratch and verifying it matches PyTorch
6. Visualizing attention weights

## 1. The Problem: Fixed-Size Bottleneck

In sequence-to-sequence models (like translation), the encoder summarizes the entire input sentence into **one fixed-size vector** (the final hidden state).

**Imagine translating a 100-word sentence by reading it once, summarizing it in one sentence, then translating from ONLY that summary.**
You'd lose most of the details.

**Attention** solves this: instead of compressing everything into one vector, the decoder can **look back at all encoder states** and decide which parts are most relevant for each output word.

For example, when translating "bank" in "I went to the bank", attention looks at the whole sentence context to determine which meaning (river bank vs financial bank) is correct.

## 2. Scaled Dot-Product Attention

The attention mechanism computes a **weighted sum of values**, where the weights come from comparing queries and keys.

```
Attention(Q, K, V) = softmax(Q @ K^T / sqrt(d_k)) @ V
```

**Q (Query):** What am I looking for?
**K (Key):** What do I have to offer?
**V (Value):** What is the actual content?

**Intuition:**
- You're searching a library (K = book titles, V = book contents)
- Your query (Q) matches against all titles
- You get back a weighted combination of all books, weighted by how relevant each title was

The `sqrt(d_k)` division prevents the dot products from getting too large (which would make softmax very peaked).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

def scaled_dot_product_attention(Q, K, V, mask=None):
    '''
    Q: (batch, heads, seq_q, d_k)
    K: (batch, heads, seq_k, d_k)
    V: (batch, heads, seq_v, d_v)   — seq_k == seq_v always

    Returns:
      output: (batch, heads, seq_q, d_v)
      weights: (batch, heads, seq_q, seq_k)
    '''
    d_k = Q.shape[-1]

    # 1. Compute raw attention scores
    scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)   # (batch, heads, seq_q, seq_k)

    # 2. Apply mask (e.g., causal mask: future tokens can't be attended to)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    # 3. Softmax → attention weights (sum to 1 along seq_k dimension)
    weights = F.softmax(scores, dim=-1)

    # 4. Weighted sum of values
    output = weights @ V   # (batch, heads, seq_q, d_v)

    return output, weights


# Test with a simple example
batch, seq_len, d_k = 1, 4, 8
Q = torch.randn(batch, 1, seq_len, d_k)
K = torch.randn(batch, 1, seq_len, d_k)
V = torch.randn(batch, 1, seq_len, d_k)

out, attn_weights = scaled_dot_product_attention(Q, K, V)
print(f'Output shape   : {out.shape}')            # (1, 1, 4, 8)
print(f'Weights shape  : {attn_weights.shape}')   # (1, 1, 4, 4)
print(f'Weights row sum: {attn_weights[0,0,0].sum():.4f}')   # should be 1.0

## 3. Self-Attention — Attending to Yourself

In **self-attention**, Q, K, and V all come from the **same sequence**.
Each token looks at all other tokens to build a context-aware representation.

**Example:** In "The animal didn't cross the street because it was too tired"
- When encoding "it", self-attention lets the model look at "animal", "street", "tired"
- It learns to attend to "animal" most (because "it" refers to "animal")

This is what makes Transformers so powerful at capturing long-range dependencies.

In [ ]:
# Self-attention: Q = K = V come from the same sequence
class SelfAttention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.d_k = embed_dim

        # Linear projections for Q, K, V
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_o = nn.Linear(embed_dim, embed_dim)  # output projection

    def forward(self, x, mask=None):
        # x: (batch, seq_len, embed_dim)
        Q = self.W_q(x)   # (batch, seq_len, d_k)
        K = self.W_k(x)
        V = self.W_v(x)

        # Add head dimension (we'll do multi-head in Notebook 12)
        Q = Q.unsqueeze(1)   # (batch, 1, seq_len, d_k)
        K = K.unsqueeze(1)
        V = V.unsqueeze(1)

        out, weights = scaled_dot_product_attention(Q, K, V, mask)
        out = out.squeeze(1)   # (batch, seq_len, d_k)
        return self.W_o(out), weights.squeeze(1)


attn = SelfAttention(embed_dim=16)
x = torch.randn(2, 5, 16)    # batch=2, seq_len=5, embed=16
out, weights = attn(x)
print(f'Input  : {x.shape}')
print(f'Output : {out.shape}')
print(f'Weights: {weights.shape}')   # (2, 5, 5) — each token attends to all others

## 4. Visualizing Attention Weights

Attention weights tell us "when processing token i, how much does the model look at token j?"
This gives us interpretability — we can see what the model focuses on.

In [ ]:
# Create a simple sequence and visualize which tokens attend to which
words = ['the', 'cat', 'sat', 'on', 'the', 'mat']
seq_len = len(words)

# Random embeddings (pretend these are word embeddings)
torch.manual_seed(7)
embed_dim = 32
x = torch.randn(1, seq_len, embed_dim)

attn = SelfAttention(embed_dim=embed_dim)
with torch.no_grad():
    out, weights = attn(x)

attn_map = weights[0].numpy()   # (seq_len, seq_len)

plt.figure(figsize=(7, 6))
im = plt.imshow(attn_map, cmap='Blues', vmin=0, vmax=attn_map.max())
plt.colorbar(im)
plt.xticks(range(seq_len), words, rotation=45)
plt.yticks(range(seq_len), words)
plt.xlabel('Keys (attending TO)')
plt.ylabel('Queries (attending FROM)')
plt.title('Self-Attention Weights\n(random — will become meaningful after training)')
plt.tight_layout()
plt.show()

print('Each row sums to 1:', np.allclose(attn_map.sum(axis=1), 1.0))

## 5. Causal (Masked) Self-Attention

In language **generation** (like GPT), when predicting token t, you should NOT be able to see tokens t+1, t+2, ... (the future).

A **causal mask** sets future positions to -infinity before softmax (so they get weight ~0).

This is called **autoregressive** attention.

In [ ]:
def causal_mask(seq_len, device='cpu'):
    '''Returns a lower-triangular mask (1 = can attend, 0 = cannot).'''
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.unsqueeze(0).unsqueeze(0)   # (1, 1, seq_len, seq_len)


seq_len = 5
mask = causal_mask(seq_len)

print('Causal mask (1=allowed, 0=blocked):')
print(mask.squeeze())

# Visualize masked attention
x = torch.randn(1, seq_len, 32)
attn = SelfAttention(32)
with torch.no_grad():
    _, weights_causal = attn(x, mask=mask)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Without mask
with torch.no_grad():
    _, weights_full = attn(x, mask=None)

ax1.imshow(weights_full[0].numpy(), cmap='Blues')
ax1.set_title('Full Attention')
ax1.set_xlabel('Key position')
ax1.set_ylabel('Query position')

ax2.imshow(weights_causal[0].numpy(), cmap='Blues')
ax2.set_title('Causal (Masked) Attention')
ax2.set_xlabel('Key position')
ax2.set_ylabel('Query position')

plt.tight_layout()
plt.show()

print('Upper triangle is all zeros (future blocked).')

In [ ]:
# Verify our implementation matches PyTorch's built-in
# PyTorch 2.0+ has F.scaled_dot_product_attention

Q = torch.randn(2, 1, 5, 8)
K = torch.randn(2, 1, 5, 8)
V = torch.randn(2, 1, 5, 8)

our_out, _ = scaled_dot_product_attention(Q, K, V)
pt_out     = F.scaled_dot_product_attention(Q, K, V)   # PyTorch built-in

print('Our output matches PyTorch:', torch.allclose(our_out, pt_out, atol=1e-5))

---
## YOUR TURN — Mini Project

**Task:** Implement attention from complete scratch (no helper functions) and use it for sequence classification.

**Steps:**
1. Implement `scaled_dot_product_attention(Q, K, V)` from scratch (no peeking at the code above!)
   - Start from the formula: `softmax(Q @ K.T / sqrt(d_k)) @ V`
2. Build an `AttentionClassifier`:
   - `nn.Embedding(vocab_size, 32)` → `SelfAttention(32)` → pool → `nn.Linear(32, num_classes)`
   - For pooling, take the **mean** of all token outputs: `out.mean(dim=1)`
3. Use the toy sentiment dataset from Notebook 10
4. Train for 500 epochs
5. Plot the attention weights for 2 positive and 2 negative examples — do you see any pattern?

**Hint:** Use the word-level vocabulary you built in Notebook 10's project.

In [ ]:
# YOUR CODE HERE

# 1. Implement scaled_dot_product_attention from scratch
def my_attention(Q, K, V, mask=None):
    # Your implementation here
    pass

# 2. AttentionClassifier
class AttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        # Your implementation here
        pass

    def forward(self, x):
        # Your implementation here
        pass

# 3. Train

# 4. Visualize attention weights